# DSAI 490 - Assignment 1: Autoencoders (AE & VAE)

### Instructions:
1. Upload `project.zip` using the cell below (or upload manually via Files panel)
2. Run all cells in order
3. GPU runtime is recommended: **Runtime > Change runtime type > GPU**

## Cell 1 — Upload & Unzip Project

In [ ]:
from google.colab import files
import os

print('Please upload project.zip ...')
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
print(f'Uploaded: {zip_name}')

In [ ]:
# Unzip the project
!unzip -o {zip_name} -d /content/
print('\nProject structure:')
!find /content/project -type f | sort

## Cell 2 — Install Requirements

In [ ]:
!pip install -r /content/project/requirements.txt -q
print('All packages installed!')

## Cell 3 — Mount Google Drive & Download Dataset

In [ ]:
from google.colab import drive
import os, shutil, kagglehub

drive.mount('/content/drive')

DRIVE_DATA_DIR = '/content/drive/MyDrive/medical-mnist'

if not os.path.exists(DRIVE_DATA_DIR):
    print('Downloading dataset from Kaggle...')
    os.environ['KAGGLEHUB_VERIFY_SSL'] = '0'
    temp_path = kagglehub.dataset_download('andrewmvd/medical-mnist')
    print('Downloaded to:', temp_path)
    shutil.copytree(temp_path, DRIVE_DATA_DIR)
    print('Saved to Google Drive.')
else:
    print('Dataset already in Drive:', DRIVE_DATA_DIR)

## Cell 4 — Load & Preprocess Data

In [ ]:
import sys
sys.path.insert(0, '/content/project')

from src.data_processing import load_and_preprocess_data

train_ds, test_ds, class_names = load_and_preprocess_data(DRIVE_DATA_DIR)
print('Classes found:', class_names)
print('Data loaded successfully!')

## Cell 5 — Build Models

In [ ]:
from src.model import build_ae, build_vae

latent_dim = 32

# Build AE
ae_model, ae_encoder, ae_decoder = build_ae(latent_dim)
ae_model.compile(optimizer='adam', loss='mse')
print('=== Autoencoder ===')
ae_model.summary()

# Build VAE
vae_encoder, vae_decoder = build_vae(latent_dim)
print('\n=== VAE Encoder ===')
vae_encoder.summary()
print('\n=== VAE Decoder ===')
vae_decoder.summary()

## Cell 6 — Train AE

In [ ]:
epochs = 5

print('Training Autoencoder (AE)...')
ae_model.fit(train_ds, epochs=epochs, validation_data=test_ds)
print('AE training complete!')

## Cell 7 — Train VAE

In [ ]:
from src.train import train_vae

print('Training Variational Autoencoder (VAE)...')
vae_encoder, vae_decoder = train_vae(vae_encoder, vae_decoder, train_ds, epochs=epochs)
print('VAE training complete!')

## Cell 8 — Reconstruct & Visualize

In [ ]:
import matplotlib.pyplot as plt
import tensorflow as tf

def plot_results(ae_model, vae_encoder, vae_decoder, dataset):
    images, _ = next(iter(dataset))
    ae_recs = ae_model(images)
    _, _, z = vae_encoder(images)
    vae_recs = vae_decoder(z)

    plt.figure(figsize=(15, 8))
    for i in range(5):
        plt.subplot(3, 5, i + 1)
        plt.imshow(images[i, :, :, 0], cmap='gray')
        plt.title('Original'); plt.axis('off')

        plt.subplot(3, 5, i + 6)
        plt.imshow(ae_recs[i, :, :, 0], cmap='gray')
        plt.title('AE Rec'); plt.axis('off')

        plt.subplot(3, 5, i + 11)
        plt.imshow(vae_recs[i, :, :, 0], cmap='gray')
        plt.title('VAE Rec'); plt.axis('off')

    plt.tight_layout()
    plt.show()

plot_results(ae_model, vae_encoder, vae_decoder, test_ds)

## Cell 9 — Latent Space Visualization

In [ ]:
import numpy as np

def plot_latent(encoder, dataset, title, is_vae=False):
    latents = []
    for images, _ in dataset.take(10):
        res = encoder(images)
        z = res[0] if is_vae else res
        latents.append(z.numpy())
    latents = np.concatenate(latents, axis=0)

    plt.figure(figsize=(8, 6))
    plt.scatter(latents[:, 0], latents[:, 1], alpha=0.5, c='steelblue')
    plt.title(f'{title} Latent Space (dim 0 vs dim 1)')
    plt.xlabel('Dim 0'); plt.ylabel('Dim 1')
    plt.show()

plot_latent(ae_encoder, test_ds, 'AE')
plot_latent(vae_encoder, test_ds, 'VAE', is_vae=True)

## Cell 10 — Generate New Samples (VAE)

In [ ]:
def generate_vae_samples(decoder, n=10, latent_dim=32):
    z_samples = tf.random.normal(shape=(n, latent_dim))
    gen_images = decoder(z_samples)

    plt.figure(figsize=(15, 3))
    for i in range(n):
        plt.subplot(1, n, i + 1)
        plt.imshow(gen_images[i, :, :, 0], cmap='gray')
        plt.axis('off')
    plt.suptitle('Generated Samples from VAE')
    plt.show()

generate_vae_samples(vae_decoder, latent_dim=latent_dim)

## Cell 11 — Denoising Test

In [ ]:
def test_denoising(model, dataset):
    images, _ = next(iter(dataset))
    noisy = tf.clip_by_value(images + 0.2 * tf.random.normal(shape=tf.shape(images)), 0.0, 1.0)
    denoised = model(noisy)

    plt.figure(figsize=(15, 5))
    for i in range(5):
        plt.subplot(2, 5, i + 1)
        plt.imshow(noisy[i, :, :, 0], cmap='gray')
        plt.title('Noisy'); plt.axis('off')

        plt.subplot(2, 5, i + 6)
        plt.imshow(denoised[i, :, :, 0], cmap='gray')
        plt.title('Denoised'); plt.axis('off')
    plt.show()

test_denoising(ae_model, test_ds)

## Cell 12 — Run Unit Tests

In [ ]:
!python -m pytest /content/project/tests/ -v